In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
from pathlib import Path
import gc
import json
import time

import torch
from PIL import Image, ImageOps
from IPython.display import display
from transformers import (
    AutoConfig,
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
    set_seed,
)


In [ ]:
# ID использует только локальный HF-кэш. Можно заменить на путь к папке модели.
MODEL_PATH = "Qwen/Qwen3-VL-4B-Instruct"
IMAGE_PATH = Path("test_apple.jpg")
CROP = "apple"  # apple или cherry

MIN_PIXELS = 256 * 256
MAX_PIXELS = 768 * 768
MAX_NEW_TOKENS = 384
CONTEXT_LIMIT = 4096
REPEATS = 3
SEED = 42

CLASSES = {
    "apple": ["apple_scab", "apple_rust", "apple_powdery_mildew"],
    "cherry": ["cherry_leaf_spot", "cherry_powdery_mildew"],
}
CROP_NAMES = {"apple": "яблоня", "cherry": "вишня"}

if CROP not in CLASSES:
    raise ValueError("CROP должен быть apple или cherry")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH.resolve())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна: проверьте Python-окружение и драйвер GPU")

DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM, GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Вычисления:", DTYPE)


In [ ]:
with Image.open(IMAGE_PATH) as source:
    image = ImageOps.exif_transpose(source).convert("RGB").copy()

print("Размер изображения:", image.size)
display(image)


In [ ]:
# Освобождение предыдущих объектов при повторном выполнении ячейки.
for name in ("inputs", "output", "generated_ids", "model", "processor"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

config = AutoConfig.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=False,
)
if config.model_type != "qwen3_vl":
    raise ValueError(f"Нужна Qwen3-VL; получена модель {config.model_type}")

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=False,
)
processor.image_processor.size = {
    "shortest_edge": MIN_PIXELS,
    "longest_edge": MAX_PIXELS,
}

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)

start = time.perf_counter()
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=False,
    use_safetensors=True,
    quantization_config=quantization,
    dtype=DTYPE,
    device_map={"": 0},
    attn_implementation="sdpa",
).eval()
torch.cuda.synchronize()
print(f"Модель загружена за {time.perf_counter() - start:.2f} с")


In [ ]:
system_prompt = (
    "Ты анализируешь видимые признаки на фото листьев растений. "
    "Фото не подтверждает возбудителя лабораторно. Не придумывай признаки, "
    "не назначай лечение и не выдавай числовую вероятность. "
    "Верни только JSON по заданной схеме, без Markdown и длинных рассуждений."
)

# Тот же observations_first и порядок полей, что в исходном эксперименте.
prompt = (
    "Сначала перечисли только непосредственно видимые признаки и их положение. "
    "Затем выбери состояние и гипотезу, поддержанную этими признаками.\n"
    f"Культура: {CROP_NAMES[CROP]}.\n"
    "Допустимые disease и alternatives: " + json.dumps(CLASSES[CROP]) + ".\n"
    'Схема: {"status": "...", "disease": null, '
    '"observations": [{"description": "...", "location": "..."}], '
    '"alternatives": [], "next_photo": null}.\n'
    "status: no_visible_symptoms, suspected_disease, other_or_unknown или insufficient_image. "
    "disease — один идентификатор только при suspected_disease, иначе null. "
    "observations — до 3 кратких признаков на русском; location — словесное положение "
    "на изображении, не точная маска. alternatives — до 2 идентификаторов; "
    "next_photo — краткий запрос другого ракурса либо null. "
    "Отсутствие видимых симптомов не доказывает здоровье. "
    "Не выбирай болезнь принудительно, если признаков недостаточно."
)

messages = [
    {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt},
    ]},
]


In [ ]:
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = processor(text=[text], images=[image], return_tensors="pt")
input_tokens = int(inputs["attention_mask"].sum().item())
visual_tokens = int((inputs["input_ids"] == model.config.image_token_id).sum().item())

if input_tokens + MAX_NEW_TOKENS > CONTEXT_LIMIT:
    raise ValueError("Превышен лимит контекста: уменьшите разрешение или длину промпта")

inputs = inputs.to(DEVICE)
print("Входные токены, включая визуальные:", input_tokens)
print("Из них визуальные:", visual_tokens)


In [ ]:
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
results = []

for repeat in range(1, REPEATS + 1):
    set_seed(SEED)
    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    generated_ids = output[0, inputs["input_ids"].shape[1]:]
    answer = processor.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    eos_ids = model.generation_config.eos_token_id
    eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
    truncated = (
        generated_ids.numel() >= MAX_NEW_TOKENS
        and int(generated_ids[-1]) not in eos_ids
    )
    result = {
        "repeat": repeat,
        "answer": answer,
        "generate_seconds": elapsed,
        "input_tokens": input_tokens,
        "output_tokens": int(generated_ids.numel()),
        "truncated": truncated,
        "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
    }
    results.append(result)

    print(f"\nПовтор {repeat}: {elapsed:.2f} с")
    print(f"Пиковая память PyTorch: {result['peak_allocated_gib']:.2f} GiB")
    if truncated:
        print("ВНИМАНИЕ: ответ достиг лимита токенов без EOS")
    try:
        print(json.dumps(json.loads(answer), ensure_ascii=False, indent=2))
    except json.JSONDecodeError:
        print("Ответ не является валидным JSON:")
        print(answer)
    del output, generated_ids

print("\nУникальных текстов ответа:", len({r["answer"] for r in results}))


In [ ]:
# Запустить после завершения работы, чтобы освободить GPU.
for name in ("inputs", "output", "generated_ids", "model", "processor"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
